In [1]:
import h5py 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
def get_target_percent(filename_path, limits=[-75, 75]):
    df = pd.read_csv(filename_path)
    df["error"] = pd.to_numeric(df["error"], errors="coerce")

    if "is_sst_trial" in df:
        marker = df["is_sst_trial"].astype(str).str.strip().str.lower()
        stop_mask = marker.isin({"1", "true", "yes", "y"})
    elif "stimulus_filename" in df:
        stop_mask = df["stimulus_filename"].astype(str).str.lower().str.contains("stop", na=False)
    else:
        stop_mask = pd.Series(False, index=df.index)

    is_sst_file = bool(stop_mask.any())
    target_df = df.loc[~stop_mask].copy() if is_sst_file else df

    n_target = target_df.loc[(target_df.error > limits[0]) & (target_df.error < limits[1])].shape[0]
    p_target = np.nan if target_df.shape[0] == 0 else (n_target / target_df.shape[0]) * 100
    label = "go stimuli without stop" if is_sst_file else "all stimuli"
    print(f"Target percent in {limits} ({label}) - {p_target:.1f} % ({n_target}/{target_df.shape[0]})")
    print(f"mean +- std - {np.nanmean(target_df.error):.0f}+-{np.nanstd(target_df.error):.0f}")
    print(f"median - {np.nanmedian(target_df.error):.0f}")

    if is_sst_file:
        stop_df = df.loc[stop_mask].copy()
        n_sst_errors = int(stop_df.error.notna().sum())
        p_sst_errors = np.nan if stop_df.shape[0] == 0 else (n_sst_errors / stop_df.shape[0]) * 100
        print(f"SST error percent - {p_sst_errors:.1f} % ({n_sst_errors}/{stop_df.shape[0]})")

In [12]:
subject = "16ED"

In [13]:
records = os.listdir(os.path.join(r"../data", subject))
records = [record for record in records if (record.find("hdf") == -1) & (record.find("tms") == -1) & (record.find("png") == -1)\
           &(record.find("txt") == -1)&(record.find("asc") == -1)]
records

['01_16ED_nofb_test.csv',
 '02_16ED_fb.csv',
 '03_16ED_fb.csv',
 '04_16ED_efb.csv',
 '05_16ED_efb.csv',
 '06_16ED_nofb_test.csv']

In [14]:
limit = 75
for record in records:
    print("--------", record, "--------")
    get_target_percent(os.path.join(r"../data", subject, record), limits=[-limit, limit])


-------- 01_16ED_nofb_test.csv --------
Target percent in [-75, 75] (all stimuli) - 25.0 % (5/20)
mean +- std - -138+-67
median - -128
-------- 02_16ED_fb.csv --------
Target percent in [-75, 75] (all stimuli) - 56.0 % (14/25)
mean +- std - -79+-75
median - -66
-------- 03_16ED_fb.csv --------
Target percent in [-75, 75] (all stimuli) - 85.0 % (34/40)
mean +- std - 15+-55
median - 16
-------- 04_16ED_efb.csv --------
Target percent in [-75, 75] (all stimuli) - 80.0 % (32/40)
mean +- std - -23+-54
median - -14
-------- 05_16ED_efb.csv --------
Target percent in [-75, 75] (all stimuli) - 80.0 % (32/40)
mean +- std - 18+-52
median - 28
-------- 06_16ED_nofb_test.csv --------
Target percent in [-75, 75] (all stimuli) - 80.0 % (24/30)
mean +- std - 20+-47
median - 17
